In [0]:
# Load the GBT validation scores
import os

from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml.functions import vector_to_array

calibration_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_gbt_v1"
)

calibration_path = (
    "/Volumes/workspace/mortgage_risk/model_artifacts/"
    "gbt_v1_calibration_v1"
)

In [0]:
# Keep probabilities away from exactly 0 and 1
clip_scores = SQLTransformer(statement="""
    SELECT *,
        CASE
            WHEN risk_score IS NULL THEN NULL
            ELSE greatest(0.000001, least(0.999999, risk_score))
        END AS safe_score
    FROM __THIS__
""")

In [0]:
# Convert the probabilities to log odds
make_log_odds = SQLTransformer(statement="""
    SELECT *,
        log(safe_score / (1.0 - safe_score)) AS log_odds
    FROM __THIS__
""")
# Prepare the calibration feature
assembler = VectorAssembler(
    inputCols=["log_odds"],
    outputCol="calibration_features"
)

In [0]:
# Define the probability adjustment
calibrator = LogisticRegression(
    featuresCol="calibration_features",
    labelCol="label_12m",
    probabilityCol="calibrated_probability",
    rawPredictionCol="calibration_raw",
    predictionCol="calibration_prediction",
    regParam=0.0,
    maxIter=100
)

In [0]:
# Fit the calibration model
calibration_pipeline = Pipeline(
    stages=[
        clip_scores,
        make_log_odds,
        assembler,
        calibrator
    ]
)

if os.path.exists(calibration_path):
    calibration_model = PipelineModel.load(calibration_path)
    print("Loaded the saved calibration model.")

else:
    calibration_model = calibration_pipeline.fit(calibration_scores)
    print("Calibration model fitted.")

Calibration model fitted.


In [0]:
slope = calibration_model.stages[-1].coefficients[0]

print("Calibration slope:", slope)

assert slope > 0, "The calibration slope must be positive."

if not os.path.exists(calibration_path):
    calibration_model.write().save(calibration_path)
    print("Calibration model saved.")

Calibration slope: 1.4558278924187658
Calibration model saved.


In [0]:
# Check the adjusted probabilities
calibrated_scores = calibration_model.transform(calibration_scores)

calibrated_scores = calibrated_scores.withColumn(
    "calibrated_risk",
    vector_to_array("calibrated_probability")[1]
)

display(
    calibrated_scores.agg(
        F.count("*").alias("rows"),
        F.avg("risk_score").alias("original_average_risk"),
        F.avg("calibrated_risk").alias("calibrated_average_risk"),
        F.avg("label_12m").alias("observed_event_rate")
    )
)

rows,original_average_risk,calibrated_average_risk,observed_event_rate
1247263,0.031491337449977334,0.004917973629501847,0.004917968383572671
